# XGBoost — genetic — prange — lowthres — lock smoking

Feature locking experiment. cgtsmok (smoking) is locked — DiCE cannot suggest smoking changes.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from dotenv import load_dotenv

load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

C:\Users\assia\L3_informatique\Stage\cf_bench\cf_outputs
True


In [2]:
batch_cfcheck_data = CF_OUTPUTS / "feature_locking_tests/XGBoost_genetic_hltprhc_2026-04-30_lock_smoking" / "genetic_annotated_hltprhc.csv"
batch_cfcheck_df = pd.read_csv(batch_cfcheck_data)


In [3]:
pd.set_option("display.max_rows", None)

In [4]:
batch_cfcheck_df = batch_cfcheck_df.drop(columns=["hltprhc", "target_risk"])

In [5]:

batch_cfcheck_df["cf_id"] = batch_cfcheck_df["cf_id"].replace({"original": "xin"})

batch_cfcheck_df = batch_cfcheck_df.rename(columns={
    "original_risk": "risk_before",
    "predicted_risk" : "predicted_risk_after",
    "meets_target_risk" : "valid",
})

batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,cf_gen_time_sec
0,0,xin,4.0,3.0,3.0,4.0,2.0,0.0,18.986591,0.0,0.025955,NaN,NaN,1.2
1,0,cf_1,NaN,NaN,NaN,NaN,NaN,NaN,17.738780,NaN,0.025955,0.002971,True,NaN
2,0,cf_1,NaN,NaN,NaN,7.0,NaN,NaN,18.974530,NaN,0.025955,0.070732,False,NaN
3,0,cf_1,NaN,NaN,NaN,NaN,NaN,NaN,18.961930,2.0,0.025955,0.007018,True,NaN


In [6]:
# correct_cf_idx
# Fix cf_id indexing - group by query_index and number counterfactuals sequentially
def fix_cf_id(df):
    # For each query_index group
    fixed_rows = []
    for query_idx, group in df.groupby('query_index', sort=False):
        group = group.copy()
        # First row is xin (original)
        xin_mask = group['cf_id'] == 'xin'
        cf_mask = ~xin_mask

        # Keep xin rows as is
        xin_rows = group[xin_mask].copy()

        # Fix cf rows
        cf_rows = group[cf_mask].copy()
        if len(cf_rows) > 0:
            # Number them cf_1, cf_2, cf_3, etc.
            cf_rows['cf_id'] = [f'cf_{i+1}' for i in range(len(cf_rows))]

        # Combine back
        fixed_rows.append(xin_rows)
        if len(cf_rows) > 0:
            fixed_rows.append(cf_rows)

    return pd.concat(fixed_rows, ignore_index=True)

batch_cfcheck_df = fix_cf_id(batch_cfcheck_df)
batch_cfcheck_df.head(30)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,cf_gen_time_sec
0,0,xin,4.0,3.0,3.0,4.0,2.0,0.0,18.986591,0.0,0.025955,NaN,NaN,1.20
1,0,cf_1,NaN,NaN,NaN,NaN,NaN,NaN,17.738780,NaN,0.025955,0.002971,True,NaN
2,0,cf_2,NaN,NaN,NaN,7.0,NaN,NaN,18.974530,NaN,0.025955,0.070732,False,NaN
3,0,cf_3,NaN,NaN,NaN,NaN,NaN,NaN,18.961930,2.0,0.025955,0.007018,True,NaN
4,0,cf_4,NaN,NaN,NaN,NaN,NaN,NaN,18.956540,2.0,0.025955,0.007018,True,NaN
5,0,cf_5,NaN,NaN,NaN,NaN,1.0,NaN,18.938780,NaN,0.025955,0.005030,True,NaN
6,0,cf_6,NaN,NaN,NaN,NaN,NaN,NaN,18.937000,7.0,0.025955,0.005873,True,NaN
7,0,cf_7,NaN,NaN,NaN,NaN,NaN,NaN,18.937000,2.0,0.025955,0.007018,True,NaN
8,0,cf_8,NaN,NaN,NaN,NaN,NaN,NaN,18.937000,1.0,0.025955,0.007468,True,NaN
9,0,cf_9,NaN,NaN,NaN,7.0,NaN,NaN,18.937000,NaN,0.025955,0.070732,False,NaN


In [7]:
# round BMI to 2 decimal places, and set to NaN if equal to xin after rounding
def round_and_clear_bmi(df):
    df = df.copy()

    # Round BMI to 2 decimals for all rows
    df['bmi'] = df['bmi'].round(2)

    # For each query_index, get the xin BMI and compare
    for query_idx in df['query_index'].unique():
        query_mask = df['query_index'] == query_idx
        xin_mask = query_mask & (df['cf_id'] == 'xin')
        cf_mask = query_mask & (df['cf_id'] != 'xin')

        # Get the xin BMI value for this query
        xin_bmi = df.loc[xin_mask, 'bmi'].values[0]

        # Set cf BMI to NaN where it equals xin BMI
        equal_mask = cf_mask & (df['bmi'] == xin_bmi)
        df.loc[equal_mask, 'bmi'] = np.nan

    return df

batch_cfcheck_df = round_and_clear_bmi(batch_cfcheck_df)
batch_cfcheck_df.head(30)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,cf_gen_time_sec
0,0,xin,4.0,3.0,3.0,4.0,2.0,0.0,18.99,0.0,0.025955,NaN,NaN,1.20
1,0,cf_1,NaN,NaN,NaN,NaN,NaN,NaN,17.74,NaN,0.025955,0.002971,True,NaN
2,0,cf_2,NaN,NaN,NaN,7.0,NaN,NaN,18.97,NaN,0.025955,0.070732,False,NaN
3,0,cf_3,NaN,NaN,NaN,NaN,NaN,NaN,18.96,2.0,0.025955,0.007018,True,NaN
4,0,cf_4,NaN,NaN,NaN,NaN,NaN,NaN,18.96,2.0,0.025955,0.007018,True,NaN
5,0,cf_5,NaN,NaN,NaN,NaN,1.0,NaN,18.94,NaN,0.025955,0.005030,True,NaN
6,0,cf_6,NaN,NaN,NaN,NaN,NaN,NaN,18.94,7.0,0.025955,0.005873,True,NaN
7,0,cf_7,NaN,NaN,NaN,NaN,NaN,NaN,18.94,2.0,0.025955,0.007018,True,NaN
8,0,cf_8,NaN,NaN,NaN,NaN,NaN,NaN,18.94,1.0,0.025955,0.007468,True,NaN
9,0,cf_9,NaN,NaN,NaN,7.0,NaN,NaN,18.94,NaN,0.025955,0.070732,False,NaN


In [8]:
int_columns = [
    "etfruit",
    "eatveg",
    "cgtsmok",
    "alcfreq",
    "slprl",
    "paccnois",
    "dosprt",
]

float_columns = [
    "bmi",
    "risk_before",
    "predicted_risk_after",
]

In [9]:
batch_cfcheck_df[int_columns] = batch_cfcheck_df[int_columns].astype("Int64")

In [10]:
batch_cfcheck_df[float_columns] = batch_cfcheck_df[float_columns].round(2)

batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,cf_gen_time_sec
0,0,xin,4,3,3,4,2,0,18.99,0,0.03,NaN,NaN,1.2
1,0,cf_1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,17.74,<NA>,0.03,0.00,True,NaN
2,0,cf_2,<NA>,<NA>,<NA>,7,<NA>,<NA>,18.97,<NA>,0.03,0.07,False,NaN
3,0,cf_3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,18.96,2,0.03,0.01,True,NaN


In [11]:
batch_cfcheck_df[int_columns] = (
    batch_cfcheck_df[int_columns]
    .astype("string")
    .fillna("")
)

In [12]:
batch_cfcheck_df = batch_cfcheck_df.replace({np.nan : ""})
batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,cf_gen_time_sec
0,0,xin,4,3,3,4,2,0,18.99,0,0.03,,,1.2
1,0,cf_1,,,,,,,17.74,,0.03,0.0,True,
2,0,cf_2,,,,7,,,18.97,,0.03,0.07,False,
3,0,cf_3,,,,,,,18.96,2,0.03,0.01,True,


In [13]:
mask = batch_cfcheck_df["cf_id"] == "xin"
# Convert to object dtype first so pandas 2.x accepts empty string
batch_cfcheck_df["risk_before"] = batch_cfcheck_df["risk_before"].astype(object)
batch_cfcheck_df.loc[mask, "risk_before"] = ""
batch_cfcheck_df.head(4)


TypeError: Invalid value '' for dtype 'float64'

In [ ]:
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# 1. Beräkna Nchanged endast för CF-rader
mask_cf = batch_cfcheck_df["cf_id"] != "xin"

batch_cfcheck_df.loc[mask_cf, "Nchanged"] = (
    batch_cfcheck_df.loc[mask_cf, feature_cols] != ""
).sum(axis=1)

# 2. Konvertera kolumnen till ren string
batch_cfcheck_df["Nchanged"] = batch_cfcheck_df["Nchanged"].astype("string")

# 3. Baseline ska vara tom
batch_cfcheck_df.loc[batch_cfcheck_df["cf_id"] == "xin", "Nchanged"] = ""

In [ ]:
batch_cfcheck_df.head(4)

In [ ]:
batch_cfcheck_df["Gower"] = ""
batch_cfcheck_df["Expected"] = ""


In [ ]:
order = ["query_index", "cf_id"] + feature_cols + ["cf_gen_time_sec", "Gower", "Nchanged", "valid", "risk_before", "predicted_risk_after", "Expected"]

batch_cfcheck_df = batch_cfcheck_df[order]
batch_cfcheck_df

# picking correct CF

### expected


In [ ]:
# Check directional constraints directly
# These features should only improve (increase) or stay the same
SHOULD_INCREASE = ["cgtsmok", "alcfreq", "dosprt"]

# These features should only improve (decrease) or stay the same
SHOULD_DECREASE = ["bmi", "etfruit", "eatveg", "slprl", "paccnois"]


print("Directional constraints:")
print(f"  Should increase (↑): {SHOULD_INCREASE}")
print(f"  Should decrease (↓): {SHOULD_DECREASE}")


In [ ]:
# Check if CFs violate directional constraints
# First, ensure numeric columns are actually numeric for comparison
numeric_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# Create a working copy with numeric values
df_check = batch_cfcheck_df.copy()
for col in numeric_cols:
    df_check[col] = pd.to_numeric(df_check[col], errors="coerce")

# Extract baseline values per query_index
baseline_values = df_check[df_check["cf_id"] == "xin"].set_index("query_index")[numeric_cols]

# Check violations for each CF
def check_violations(row):
    if row["cf_id"] == "xin":
        return ""

    baseline = baseline_values.loc[row["query_index"]]
    violations = []

    # Check features that should increase
    for feat in SHOULD_INCREASE:
        if pd.notna(row[feat]) and pd.notna(baseline[feat]):
            if row[feat] < baseline[feat]:
                violations.append(f"{feat}↓")

    # Check features that should decrease
    for feat in SHOULD_DECREASE:
        if pd.notna(row[feat]) and pd.notna(baseline[feat]):
            if row[feat] > baseline[feat]:
                violations.append(f"{feat}↑")

    return "NO: " + ", ".join(violations) if violations else True

batch_cfcheck_df["Expected"] = df_check.apply(check_violations, axis=1)
batch_cfcheck_df.head(10)


### is valid

In [ ]:
# --- Select exactly one CF per query_index (random valid without violations) ---

# Split baseline and CF rows
df_xin = batch_cfcheck_df[batch_cfcheck_df["cf_id"] == "xin"]
df_cf  = batch_cfcheck_df[batch_cfcheck_df["cf_id"] != "xin"]

def select_random_cf(group):
    """Select one random CF that is valid and has no violations (Expected is empty)."""
    # First try: valid AND no violations
    good_cfs = group[(group["valid"] == True) & (group["Expected"] == "")]
    if len(good_cfs) > 0:
        return good_cfs.sample(n=1, random_state=42)

    # Fallback: any valid CF
    valid_cfs = group[group["valid"] == True]
    if len(valid_cfs) > 0:
        return valid_cfs.sample(n=1, random_state=42)

    # Last resort: first CF
    return group.iloc[[0]]

# Select one random CF per query_index
df_cf_selected = df_cf.groupby("query_index", group_keys=False).apply(select_random_cf).reset_index(drop=True)

# Combine baseline + selected CF
batch_cfcheck_df = pd.concat([df_xin, df_cf_selected], ignore_index=True)

# Ensure xin appears before CF for each query_index
batch_cfcheck_df["is_xin"] = (batch_cfcheck_df["cf_id"] == "xin").astype(int)
batch_cfcheck_df = (
    batch_cfcheck_df
    .sort_values(["query_index", "is_xin"], ascending=[True, False])
    .drop(columns="is_xin")
)
batch_cfcheck_df

In [ ]:
# Print human-readable recommendations for each person
# (same output as DiceRecommender.format_query, written inline to avoid TF import)
feature_cols = ['etfruit', 'eatveg', 'cgtsmok', 'alcfreq', 'slprl', 'paccnois', 'bmi', 'dosprt']

for query_idx in sorted(batch_cfcheck_df['query_index'].unique()):
    block = batch_cfcheck_df[batch_cfcheck_df['query_index'] == query_idx]
    xin   = block[block['cf_id'] == 'xin'].iloc[0]
    cfs   = block[block['cf_id'] != 'xin']

    print(f"
=== Person {query_idx} ===")
    print("Original:", {f: xin[f] for f in feature_cols if xin[f] != ""})
    print(f"Risk before: {xin.get('risk_before', '?')}")

    if cfs.empty:
        print("  No counterfactuals.")
        continue

    for _, cf in cfs.iterrows():
        changes = [(f, xin[f], cf[f]) for f in feature_cols
                   if cf[f] != "" and cf[f] != xin[f]]
        status = "VALID" if cf["valid"] == True else "invalid"
        risk   = cf.get("predicted_risk_after", "?")
        print(f"  {cf['cf_id']} [{status}] risk={risk}  changes: {changes}")
